In [8]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_NAME"),
    base_url=os.getenv("OPENAI_API_BASE"),
    temperature=0
    )

## message inputs

Adding memory to a chat model provides a simple example. Chat models accept a list of messages as input and output a message. LangGraph includes a built-in MessagesState that we can use for this purpose.

Below, we:

- Define the graph state to be a list of messages;
- Add a single node to the graph that calls a chat model;
- Compile the graph with an in-memory checkpointer to store messages between runs.

In [49]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)  # 使用MessagesState作为状态，创建一个状态图


# Define the function that calls the model
def call_model(state: MessagesState): #定义一个函数，输入是MessagesState，输出也是MessagesState
    response = llm.invoke(state["messages"])
    # Update message history with response:
    return {"messages": response}


# Define the (single) node in the graph
workflow.add_edge(START, "model") #添加从开始到model节点的边
workflow.add_node("model", call_model) #添加model节点，使用call_model函数

"""
StateGraph是LangGraph中用于构建状态图的核心类,主要方法包括:

初始化方法：
workflow = StateGraph(state_schema)
state_schema: 定义状态图的schema类型(如MessagesState)
节点管理方法：
workflow.add_node(name, func)
name: 节点名称
func: 节点处理函数
边管理方法：
workflow.add_edge(start_node, end_node)
start_node: 起始节点
end_node: 目标节点
条件边方法：
workflow.add_conditional_edges(
    start_node,
    condition_func,
    edge_mapping
)
start_node: 起始节点
condition_func: 条件判断函数
edge_mapping: 条件到目标节点的映射
入口设置方法：
workflow.set_entry_point(node_name)
node_name: 入口节点名称
出口设置方法：
workflow.set_finish_point(node_name)
node_name: 出口节点名称
编译方法：
app = workflow.compile(checkpointer=memory)
checkpointer: 可选的记忆系统(如MemorySaver)
状态更新方法：
new_state = workflow.update_state(current_state)
current_state: 当前状态
返回更新后的状态
"""

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory) # StateGraph编译为app

In [47]:
config = {"configurable": {"thread_id": "abc123"}} ## 增加该对话的thread_id，以识别是哪一个对话；

In [52]:
query = "Hi! I'm Bob."

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()  # 从output中获取最后一个消息，并打印出来，pretty_print()会打印出包含所有消息状态的输出；
output["messages"][-2].pretty_print()
print(output)

================================== Ai Message ==================================

Hi Bob! It's nice to meet you. How can I assist you today? 😊
================================ Human Message =================================

Hi! I'm Bob.
{'messages': [HumanMessage(content="Hi! I'm Bob.", additional_kwargs={}, response_metadata={}, id='8d6d718e-ca6b-4f3b-8086-0e59aed4d3a9'), AIMessage(content='Hi Bob! Nice to meet you. How can I assist you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 9, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-459ffb58-945a-4e6e-b18d-ac7e9d8052af-0', usage_metadata={'input_tokens': 9, 'output_tokens': 17, 'total_tokens': 26, 'input_token_details': {}, 'output_token_d

In [27]:
query = "What's my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()
output["messages"][-2].pretty_print()
output["messages"][-3].pretty_print()
output["messages"][-4].pretty_print()

================================== Ai Message ==================================

You just told me your name is Bob! 😊 Hi again, Bob! How can I help you today?
================================ Human Message =================================

What's my name?
================================== Ai Message ==================================

Hi, Bob! Great to meet you. 😊 How can I assist you today? Let me know what’s on your mind!
================================ Human Message =================================

Hi! I'm Bob.


In [23]:
query = "What's my name?"
config = {"configurable": {"thread_id": "abc234"}}

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()
output["messages"][-2].pretty_print()


================================== Ai Message ==================================

I don’t have access to personal information, so I don’t know your name. If you’d like, you can tell me, and I’ll use it in our conversation! 😊
================================ Human Message =================================

What's my name?


## dictionary inputs

LangChain runnables often accept multiple inputs via separate keys in a single dict argument. A common example is a prompt template with multiple parameters.

Whereas before our runnable was a chat model, here we chain together a prompt template and chat model.

In [54]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer in {language}."),
        MessagesPlaceholder(variable_name="messages"), # 构建含有动态内容的prompt模版，动态内容为MessagesPlaceholder存储和传递的消息；
    ]
)

runnable = prompt | llm

For this scenario, we define the graph state to include these parameters (in addition to the message history). We then define a single-node graph in the same way as before.

Note that in the below state:

- Updates to the messages list will append messages;
- Updates to the language string will overwrite the string.

In [39]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict


class State(TypedDict): # 定义一个State类型的字典，包含两个固定键，一个是messages，一个 是language；
    messages: Annotated[Sequence[BaseMessage], add_messages] # Annotated是一个类型注解，用于添加元数据，这里用于添加add_messages元数据；
    language: str


workflow = StateGraph(state_schema=State)


def call_model(state: State):
    response = runnable.invoke(state) # state作为输入，调用runnable，返回response；
    # Update message history with response:
    return {"messages": [response]}


workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [40]:
config = {"configurable": {"thread_id": "abc345"}}

input_dict = {
    "messages": [HumanMessage("Hi, I'm Bob.")],
    "language": "Spanish",
}
output = app.invoke(input_dict, config)
output["messages"][-1].pretty_print()
output["messages"][-2].pretty_print()
print(output)

================================== Ai Message ==================================

¡Hola, Bob! ¿Cómo estás? ¿En qué puedo ayudarte hoy?
================================ Human Message =================================

Hi, I'm Bob.
{'messages': [HumanMessage(content="Hi, I'm Bob.", additional_kwargs={}, response_metadata={}, id='d693d527-db79-430f-bc1d-48972cac0828'), AIMessage(content='¡Hola, Bob! ¿Cómo estás? ¿En qué puedo ayudarte hoy?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 13, 'total_tokens': 33, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-bae2eb83-a33f-4e53-ade7-17bd0b2b9fbf-0', usage_metadata={'input_tokens': 13, 'output_tokens': 20, 'total_tokens': 33, 'input_token_details': {}, 'output_token_details':

## route between sub-chains

Routing allows you to create non-deterministic chains where the output of a previous step defines the next step. Routing can help provide structure and consistency around interactions with models by allowing you to define states and use information related to those states as context to model calls.

There are two ways to perform routing:

- Conditionally return runnables from a RunnableLambda (recommended)
- Using a RunnableBranch (legacy)

In [78]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
"""
ChatPromptTemplate：专门用于聊天场景，支持结构化消息格式（如system、human、ai等角色）
PromptTemplate：通用模板，处理简单的字符串替换
"""

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `LangChain`, `DeepSeek`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | llm
    | StrOutputParser()
)

chain.invoke({"question": "how do I call DeepSeek?"})

'DeepSeek'

In [69]:
langchain_chain = PromptTemplate.from_template(
    """You are an expert in langchain. \
Always answer questions starting with "As Harrison Chase told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | llm
deepseek_chain = PromptTemplate.from_template(
    """You are an expert in deepseek. \
Always answer questions starting with "As Dario Amodei told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | llm
general_chain = PromptTemplate.from_template(
    """Respond to the following question:

Question: {question}
Answer:"""
) | llm

In [70]:
def route(info):  # Recommend Method
    if "deepseek" in info["topic"].lower():
        return deepseek_chain
    elif "langchain" in info["topic"].lower():
        return langchain_chain
    else:
        return general_chain

In [81]:
from langchain_core.runnables import RunnableLambda

full_chain = {"topic": chain, "question": lambda x: x["question"]} | RunnableLambda(  # chain执行定义的runable以question输入运行，question执行lambda函数，返回弟字典中“question”的值
    route
)

In [72]:
full_chain.invoke({"question": "how do I use deepseek?"})

AIMessage(content='As Dario Amodei told me, DeepSeek is a powerful tool designed for advanced data analysis and pattern recognition. To use DeepSeek effectively, follow these steps:\n\n1. **Installation**: First, ensure you have the necessary dependencies installed, such as Python and relevant libraries. You can install DeepSeek via pip or by cloning the repository from its official source.\n\n2. **Data Preparation**: Organize your dataset in a format compatible with DeepSeek. This typically involves cleaning and preprocessing your data to ensure it’s ready for analysis.\n\n3. **Configuration**: Set up the configuration file or parameters according to your specific use case. This includes defining the model architecture, hyperparameters, and any specific settings for your analysis.\n\n4. **Training**: If you’re using DeepSeek for machine learning tasks, train the model on your dataset. This involves feeding the data into the model and allowing it to learn patterns and relationships.\n\

In [73]:
full_chain.invoke({"question": "how do I use LangChain?"})

AIMessage(content='As Harrison Chase told me, using LangChain involves several steps to effectively build and deploy language model-powered applications. Here\'s a high-level guide to get you started:\n\n1. **Install LangChain**: First, you need to install the LangChain library. You can do this using pip:\n   ```bash\n   pip install langchain\n   ```\n\n2. **Set Up Your Environment**: Ensure you have the necessary API keys for any services you plan to use, such as OpenAI, Hugging Face, or other LLM providers.\n\n3. **Choose a Language Model**: LangChain supports various language models. You can start with OpenAI\'s GPT models, for example:\n   ```python\n   from langchain.llms import OpenAI\n   llm = OpenAI(api_key="your-openai-api-key")\n   ```\n\n4. **Create Chains**: LangChain allows you to create chains of operations. For example, you can create a simple chain that takes user input and generates a response:\n   ```python\n   from langchain.chains import LLMChain\n   from langchain.

In [74]:
full_chain.invoke({"question": "whats 2 + 2"})

AIMessage(content='The answer to 2 + 2 is **4**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 20, 'total_tokens': 32, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 20}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-c52bd725-3569-4488-81d2-6dcfda80bf17-0', usage_metadata={'input_tokens': 20, 'output_tokens': 12, 'total_tokens': 32, 'input_token_details': {}, 'output_token_details': {}})

## using a RuannableBranch

A RunnableBranch is a special type of runnable that allows you to define a set of conditions and runnables to execute based on the input. It does not offer anything that you can't achieve in a custom function as described above, so we recommend using a custom function instead.


In [83]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "deepseek" in x["topic"].lower(), deepseek_chain),
    (lambda x: "langchain" in x["topic"].lower(), langchain_chain),
    general_chain,
)
full_chain = {"topic": chain, "question": lambda x: x["question"]} | branch
full_chain.invoke({"question": "how do I use deepseek?"})

AIMessage(content="As Dario Amodei told me, DeepSeek is a powerful tool designed for advanced data analysis and exploration. To use DeepSeek effectively, follow these steps:\n\n1. **Installation**: First, ensure you have the necessary dependencies installed. You can typically install DeepSeek via pip or another package manager. Run the command:\n   ```\n   pip install deepseek\n   ```\n\n2. **Data Preparation**: Prepare your dataset in a format compatible with DeepSeek, such as CSV, JSON, or a database connection. Ensure your data is clean and well-structured.\n\n3. **Initialization**: Import DeepSeek into your Python environment and initialize it with your dataset:\n   ```python\n   from deepseek import DeepSeek\n   ds = DeepSeek(data_path='your_dataset.csv')\n   ```\n\n4. **Exploration**: Use DeepSeek's built-in functions to explore your data. For example:\n   ```python\n   ds.summary()  # Provides a summary of the dataset\n   ds.visualize()  # Generates visualizations for better und

## Routing by semantic similarity

One especially useful technique is to use embeddings to route a query to the most relevant prompt. Here's an example.

In [87]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = HuggingFaceEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)


def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    print(PromptTemplate.from_template(most_similar))
    return PromptTemplate.from_template(most_similar)


chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

In [88]:
print(chain.invoke("What's a black hole"))

Using PHYSICS
input_variables=['query'] input_types={} partial_variables={} template="You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner. When you don't know the answer to a question you admit that you don't know.\n\nHere is a question:\n{query}"
A black hole is a region in space where gravity is so strong that nothing—not even light—can escape its pull. It forms when a massive star collapses under its own gravity at the end of its life, creating a singularity (a point of infinite density) surrounded by an event horizon. The event horizon is the "boundary" beyond which nothing can escape. Black holes are fascinating because they warp space and time, and studying them helps us understand gravity, quantum mechanics, and the universe's structure. Let me know if you'd like more details!


## creat a dynamic(self constructing) chain

Sometimes we want to construct parts of a chain at runtime, depending on the chain inputs (routing is the most common example of this). We can create dynamic chains like this using a very useful property of RunnableLambda's, which is that if a RunnableLambda returns a Runnable, that Runnable is itself invoked. Let's see an example.

In [89]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnablePassthrough, chain

contextualize_instructions = """Convert the latest user question into a standalone question given the chat history. Don't answer the question, return the question and nothing else (no descriptive text)."""
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_instructions),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ]
)
contextualize_question = contextualize_prompt | llm | StrOutputParser()

qa_instructions = (
    """Answer the user question given the following context:\n\n{context}."""
)
qa_prompt = ChatPromptTemplate.from_messages(
    [("system", qa_instructions), ("human", "{question}")]
)


@chain
def contextualize_if_needed(input_: dict) -> Runnable:
    if input_.get("chat_history"):
        # NOTE: This is returning another Runnable, not an actual output.
        return contextualize_question ## 如果有“chat_history”,返回contextualize_question增加pipe
    else:
        return RunnablePassthrough() | itemgetter("question")


@chain
def fake_retriever(input_: dict) -> str:
    return "egypt's population in 2024 is about 111 million"


full_chain = (
    RunnablePassthrough.assign(question=contextualize_if_needed).assign(
        context=fake_retriever
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

full_chain.invoke(
    {
        "question": "what about egypt",
        "chat_history": [
            ("human", "what's the population of indonesia"),
            ("ai", "about 276 million"),
        ],
    }
)

'As of 2024, the population of Egypt is approximately **111 million**.'